# MVP - Data Engineering
## Funções comuns ao notebook silver e gold

In [0]:
from pyspark.sql.functions import col, count as count_all, sum as spark_sum, when, isnull
from pyspark.sql.window import Window

In [0]:
def schema_tables(schema):
    """Retorna a lista de nomes de tabelas em um schema do Unity Catalog.
    Usa a variável global `spark` disponível no notebook.
    """
    tables = spark.sql(f"SHOW TABLES IN {schema}").collect()
    return [row.tableName for row in tables]

In [0]:
def get_columns_to_model():
    """Retorna o conjunto de colunas que pertencem ao modelo estrela."""
    return {
        "mql_id", "customer_id", "seller_id", "product_id", "order_id",
        "first_contact_date", "origin",
        "won_date", "sales_cycle",
        "customer_city", "customer_state",
        "seller_city", "seller_state",
        "purchase_state",
        "price", "sales_value", "commission", "quantity",
        "product_category_name", "category",
        "order_purchase_timestamp", "order_date",
        "year", "month", "quarter", "day_of_week",
        "order_status"
    }

In [0]:
def get_pks(schema):
    """Retorna as primary keys das tabelas da camada silver ou gold."""
    pk_silver = {
        "customers": "customer_id",
        "sellers": "seller_id",
        "marketing_qualified_leads": "mql_id",
        "closed_deals": "mql_id",
        "products": "product_id",
        "orders": "order_id",
        "order_items": ["order_id", "product_id", "seller_id"],
    }
    pk_gold = {
        "dim_leads": "mql_id",
        "dim_sellers": "seller_id",
        "dim_products": "product_id",
        "dim_dates": "order_date",
        "fato_vendas": ["order_id", "product_id", "seller_id"],
    }
    return pk_silver if schema == "silver" else pk_gold

In [0]:
def get_fks(layer):
    """Retorna os mapeamentos de FKs para verificação de integridade referencial.

    Para bronze/silver, os relacionamentos são os mesmos (mesmos nomes de tabela e colunas).
    Para gold, verifica FKs da fato_vendas → dimensões.

    Retorna lista de tuplas: (tabela, fk_col, tabela_referencia, col_referencia).
    """
    fk_bronze_silver = [
        # order_items → orders, products, sellers
        ("order_items", "order_id", "orders", "order_id"),
        ("order_items", "product_id", "products", "product_id"),
        ("order_items", "seller_id", "sellers", "seller_id"),
        # orders → customers
        ("orders", "customer_id", "customers", "customer_id"),
        # closed_deals → sellers, marketing_qualified_leads
        ("closed_deals", "seller_id", "sellers", "seller_id"),
        ("closed_deals", "mql_id", "marketing_qualified_leads", "mql_id"),
    ]
    fk_gold = [
        # NÃO inclui mql_id → dim_leads: o valor "unknown" é intencional
        # (sellers sem lead rastreado recebem COALESCE(c.mql_id, "unknown"))
        ("fato_vendas", "product_id", "dim_products", "product_id"),
        ("fato_vendas", "seller_id", "dim_sellers", "seller_id"),
        ("fato_vendas", "order_date", "dim_dates", "order_date"),
    ]
    if layer in ("bronze", "silver"):
        return fk_bronze_silver
    return fk_gold

As funções find_nulls e find_duplicates abaixo identificam problemas de qualidade na camada bronze antes da carga para silver. find_duplicates verifica tanto linhas totalmente duplicadas quanto violações de PK, comparando total de linhas vs. distintos.

In [0]:
def find_duplicates(df, schema, table_name, num_nulls, problems_found, duplicates_set):
    """Detecta linhas duplicadas e violações de PK no dataframe."""
    df_cols = [c for c in df.columns if c in get_columns_to_model()]
    total_rows = df.count()
    distinct_rows = df.select(df_cols).distinct().count()
    duplicate_count = total_rows - distinct_rows

    w = Window.partitionBy([col(c) for c in df_cols])
    duplicated_rows = df.withColumn("_dup_count", count_all("*").over(w)) \
        .filter(col("_dup_count") > 1) \
        .drop("_dup_count")

    pk_col = get_pks(schema).get(table_name)
    pk_dup_count = 0
    if pk_col:
        pk_cols_list = pk_col if isinstance(pk_col, list) else [pk_col]
        if all(c in df_cols for c in pk_cols_list):
            pk_dup_count = total_rows - df.select(pk_cols_list).distinct().count()

    if duplicate_count > 0 or pk_dup_count > 0:
        if num_nulls == 0:
            print(f"\n{'='*60}")
            print(f"Table: {table_name}")

        duplicates_set.add(table_name)
        print(f"\nDuplicados:")
        print(f"Linhas: {total_rows} | Duplicados: {duplicate_count} | PK duplicados: {pk_dup_count}")

        print(f"\nLinhas duplicadas em {table_name}:")
        duplicated_rows.limit(20).show(truncate=False)

        problems_found += 1

    return problems_found, duplicates_set

In [0]:
def find_nulls(df, table_name, problems_found, nulls_dict):
    """Conta valores nulos por coluna (apenas colunas do modelo estrela)."""
    columns = [c for c in df.columns if c in get_columns_to_model()]

    nulls_count = df.select([
        spark_sum(when(isnull(c), 1).otherwise(0)).alias(c) for c in columns
    ]).collect()[0]

    num_nulls = 0
    for col_name, null_count in nulls_count.asDict().items():
        if null_count > 0:
            if table_name not in nulls_dict.keys():
                nulls_dict[table_name] = [col_name]
            else:
                nulls_dict[table_name].append(col_name)

            if num_nulls == 0:
                print(f"\n{'='*60}")
                print(f"Table: {table_name}")
                print(f"\nNulls por coluna:")
            num_nulls += null_count
            print(f"  {col_name}: {null_count}")

            problems_found += 1

    return num_nulls, problems_found, nulls_dict

In [0]:
def check_referential_integrity(df, layer, table_name, catalog):
    """Verifica integridade referencial: toda FK da tabela deve existir na tabela de referência.

    Usa LEFT ANTI JOIN para encontrar órfãos (FKs sem correspondência).
    Retorna o número de problemas encontrados (0 = íntegro).
    """
    problems = 0
    fk_mappings = get_fks(layer)

    # Filtrar apenas os mapeamentos para esta tabela
    table_fks = [(fk_col, ref_table, ref_col)
                 for (tbl, fk_col, ref_table, ref_col) in fk_mappings
                 if tbl == table_name]

    if not table_fks:
        return problems

    for fk_col, ref_table, ref_col in table_fks:
        ref_full = f"{catalog}.{layer}.{ref_table}"
        ref_df = spark.table(ref_full)
        orphans = df.join(ref_df, df[fk_col] == ref_df[ref_col], "left_anti")
        count = orphans.count()
        if count > 0:
            problems = 1
            print(f"\n⚠️ {count} linhas em {table_name} com {fk_col} sem correspondência em {ref_table}")
            orphans.select(fk_col).limit(10).show(truncate=False)
        else:
            print(f"\n✅ Integridade referencial OK: {table_name}.{fk_col} → {ref_table}.{ref_col}")


    return problems

In [0]:
def check_negatives(df, layer, catalog):
    """Verifica se há valores negativos em colunas numéricas de métricas.

    Na camada gold, fato_vendas já exclui pedidos canceled/unavailable — verifica sales_value diretamente.
    Na camada bronze, order_items não tem order_status — faz JOIN com orders para filtrar pedidos válidos
    e verifica price < 0 antes da agregação (captura negativos individuais que poderiam ser mascarados pela soma).
    """
    problems = 0
    the_col = "price" if layer == "bronze" else "sales_value"

    if layer == "bronze":
        orders = spark.table(f"{catalog}.bronze.orders")
        df = df.join(orders, "order_id", "inner") \
               .filter(~col("order_status").isin("canceled", "unavailable"))

    negatives = df.filter(col(the_col) < 0)
    count = negatives.count()
    if count > 0:
        problems = 1
        print(f"\n⚠️ {count} linhas com {the_col} < 0")
        negatives.select("order_id", the_col).limit(10).show(truncate=False)
    else:
        print(f"\n✅ Nenhum valor negativo em {the_col}")

    return problems